# FEMI-CROSS-MODEL-1 v2 — frozen cross-model replication

48 selected cases (15 distinct ContractNLI hypotheses + 33 FEVEROUS distinct claims) × three evidence regimes × two wordings × three models = **864 calls**.

**Only cells explicitly marked PAID make OpenRouter calls.** The full source transfer archive and frozen 288 base prompts are already bundled. Do **not** run a builder, reselect cases, or change prompts once the first live response has been collected.


## 1. Upload and extract packet (OFFLINE)
Upload `FEMI_CROSS_MODEL_V2_20261108.zip` from ChatGPT. If resuming, keep the existing extracted folder and don't overwrite it; restore `runs/cross_model/results` from your saved checkpoint before making calls.

In [ ]:
from google.colab import files
from pathlib import Path
import os, subprocess, zipfile, json, sys

packet_name = 'FEMI_CROSS_MODEL_V2_20261108.zip'
pkt = Path('/content') / packet_name
if not pkt.is_file():
    uploaded = files.upload()
    assert packet_name in uploaded, f'Upload {packet_name} first'

ROOT = Path('/content/FEMI_CROSS_MODEL_V2_20261108')
if not (ROOT / 'runs/cross_model/FREEZE.json').exists():
    with zipfile.ZipFile(pkt) as z:
        z.extractall('/content')
assert (ROOT / 'runs/cross_model/manifest.jsonl').is_file()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT/'src'))
print('Working directory:', os.getcwd())
print('Frozen experiment:', (ROOT/'runs/cross_model/FREEZE.json').exists())


## 2. Software checks and frozen-input inspection (OFFLINE)
Expect **9** passing tests, 288 base prompts and 864 model-expanded requests. The first test intentionally rebuilds the source bank into a temporary directory to check determinism; it does **not** change `runs/cross_model`.

In [ ]:
subprocess.run(['python','-m','unittest','discover','-s','tests','-v'], check=True)
subprocess.run(['python','src/run_and_score.py','preflight','--dir','runs/cross_model'], check=True)
subprocess.run(['python','src/run_and_score.py','run','--dir','runs/cross_model', '--models','models.cross.v2.json'],check=True)
fr=json.loads(Path('runs/cross_model/FREEZE.json').read_text())
print('Selected by domain:', {d:v['labels'] for d,v in fr['selected_by_domain'].items()})
print('Manifest SHA-256:',fr['manifest_sha256'])


## 3. Review the model configuration (OFFLINE)
This pins GPT-OSS and Llama to Groq, and Gemini to Google AI Studio with fallbacks disabled. Model IDs may become unavailable; in that event **STOP** and preserve the frozen packet instead of silently substituting a different model or provider.

In [ ]:
models=json.loads(Path('models.cross.v2.json').read_text())['models']
for m in models:
    print(m['name'], '|',m['model_id'], '|',m['provider'])

# Optional: mount Google Drive for automatic backups after each completed batch.
# Set True to enable backups to MyDrive/FEMI_CROSS_MODEL_V2_CHECKPOINTS.
USE_DRIVE_BACKUP = False
BACKUP_DIR = None
if USE_DRIVE_BACKUP:
    from google.colab import drive
    drive.mount('/content/drive')
    BACKUP_DIR = Path('/content/drive/MyDrive/FEMI_CROSS_MODEL_V2_CHECKPOINTS')
    BACKUP_DIR.mkdir(parents=True,exist_ok=True)
print('Backup directory:', BACKUP_DIR)


## 4. PAID — first 24 planned calls (transport and schema gate)
In **Colab Secrets**, set `OPENROUTER_API_KEY` and enable notebook access. This batch is part of the frozen study, not a throwaway pilot. It makes eight calls per model due to request interleaving. Do not inspect behavioral effects or revise prompts after this checkpoint.

In [ ]:
from google.colab import userdata
key = userdata.get('OPENROUTER_API_KEY')
assert isinstance(key,str) and key.strip(), 'Set OPENROUTER_API_KEY in Colab Secrets'
os.environ['OPENROUTER_API_KEY']=key
del key

subprocess.run([
    'python','src/run_and_score.py','run',
    '--dir','runs/cross_model','--models','models.cross.v2.json',
    '--execute','--max-new','24','--pace','0.85'
],check=True)


## 5. Check transport and response format only (OFFLINE)
This shows provider and JSON-validity counts, not unauthorized-decision rates. If a model produces only invalid completions, stop and discuss the incompatibility rather than quietly modifying the decoding settings.

In [ ]:
from collections import Counter, defaultdict
from build_transfer import read_jsonl
from run_and_score import parse_decision
responses_path=Path('runs/cross_model/results/responses.jsonl')
responses=list(read_jsonl(responses_path))
counts=Counter(); invalids=Counter(); providers=defaultdict(Counter)
for r in responses:
    m=r['model_name'];counts[m]+=1
    invalids[m]+=int(parse_decision(r['result']) is None)
    providers[m][str(r['result'].get('provider') or 'UNREPORTED')]+=1
print('Responses:',len(responses),'of 864')
for model,n in sorted(counts.items()):
    print(model, '| total',n,'| invalid JSON/verdict',invalids[model], '| providers',dict(providers[model]))
assert len(responses)<=864
assert all(invalids[m]<n for m,n in counts.items()), 'Some models are entirely schema-invalid. STOP; preserve data.'


## 6. PAID — finish remaining calls, checkpointing every 72 new responses
This continues the **same** 864-call study. The runner validates all saved request hashes on every resume and refuses model-config drift. If a network or provider error occurs, preserve the directory; rerun the cell when the problem is resolved.

If `USE_DRIVE_BACKUP=True`, a copy of `responses.jsonl` and `MODEL_FREEZE.json` is saved after each completed chunk. Otherwise the run is stored only in the active Colab runtime, so download a checkpoint if the session might disconnect.

In [ ]:
import shutil
MAX_TOTAL=864
while True:
    count=sum(1 for _ in read_jsonl('runs/cross_model/results/responses.jsonl'))
    if count>=MAX_TOTAL: break
    print(f'Collected {count}/{MAX_TOTAL}; next resumable batch')
    subprocess.run([
        'python','src/run_and_score.py','run',
        '--dir','runs/cross_model','--models','models.cross.v2.json',
        '--execute','--max-new','72','--pace','0.85'
    ],check=True)
    if BACKUP_DIR is not None:
        for name in ('responses.jsonl','MODEL_FREEZE.json'):
            src=Path('runs/cross_model/results')/name
            if src.exists(): shutil.copy2(src,BACKUP_DIR/name)
print('Complete:',sum(1 for _ in read_jsonl('runs/cross_model/results/responses.jsonl')))


## 7. Full analysis and export (OFFLINE)
The analyzer will **refuse** incomplete or mismatched raw responses. After successful analysis, export the whole frozen run directory including private cases and raw outputs, and share the ZIP for independent review.

In [ ]:
subprocess.run(['python','src/run_and_score.py','analyze','--dir','runs/cross_model','--models','models.cross.v2.json'],check=True)
subprocess.run(['python','src/cross_analysis.py','--dir','runs/cross_model','--models','models.cross.v2.json'],check=True)

report=json.loads(Path('runs/cross_model/results/CROSS_MODEL_REPORT.json').read_text())
print('Analyzed responses:',report['n_responses'])
print('Reported cost, USD:',report['reported_cost_usd_by_model'])
print('Primary absent-wording contrasts:',{k:v['observed'] for k,v in report['paired_wording_contrasts'].items() if k.startswith('absent|')})

archive=shutil.make_archive('/content/FEMI_CROSS_MODEL_V2_RESULTS','zip',root_dir='runs',base_dir='cross_model')
print('ZIP created:',archive,'bytes:',Path(archive).stat().st_size)
files.download(archive)
